In [1]:
# Author: Arthur Prigent
# Email: arthur.prigent@univ-brest.fr

In [2]:
import matplotlib.patches as mpatches
from scipy.stats import pearsonr
import matplotlib.ticker as mticker
import cartopy.crs as ccrs
import cartopy
import matplotlib
import scipy
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

import matplotlib.colors as mcolors

from datetime import datetime, timedelta
import cartopy.feature as cfeature
import numpy as np
import xarray as xr
import glob
import matplotlib.pyplot as plt
import netCDF4
import string
from scipy.spatial import cKDTree
from datetime import timedelta
import gsw

### PSAL

In [3]:
ds_all_psal = xr.open_dataset('/data0/user/aprigent/PROCESSED/level_2/profiles_QC_PSAL_new.nc',decode_times=False)

### TEMP

In [4]:
ds_all_temp = xr.open_dataset('/data0/user/aprigent/PROCESSED/level_2/profiles_QC_TEMP_new.nc',decode_times=False)

In [5]:
def build_key(ds):
    # safer precision: avoid too coarse rounding
    time   = np.char.mod('%.5f', ds.time.values)
    lat    = np.char.mod('%.5f', ds.latitude.values)
    lon    = np.char.mod('%.5f', ds.longitude.values)
    source = ds.source.values.astype(str)

    key = np.char.add(time, "_")
    key = np.char.add(key, lat)
    key = np.char.add(key, "_")
    key = np.char.add(key, lon)
    key = np.char.add(key, "_")
    key = np.char.add(key, source)

    return key


def drop_duplicate_profiles(ds):
    key = build_key(ds)
    _, unique_idx = np.unique(key, return_index=True)
    n_before = ds.profile.size
    ds_deduped = ds.isel(profile=unique_idx)
    n_dropped = n_before - ds_deduped.profile.size
    print(f"Dropped {n_dropped} duplicate profiles, {ds_deduped.profile.size} remaining")
    return ds_deduped

ds_temp_test = drop_duplicate_profiles(ds_all_temp)
ds_psal_test = drop_duplicate_profiles(ds_all_psal)

Dropped 0 duplicate profiles, 368849 remaining
Dropped 0 duplicate profiles, 346161 remaining


In [6]:
# --- Find common profiles ---
key_temp = build_key(ds_all_temp)
key_psal = build_key(ds_all_psal)

common_keys = np.intersect1d(key_temp, key_psal)

ds_temp_common = ds_all_temp.isel(profile=np.isin(key_temp, common_keys))
ds_psal_common = ds_all_psal.isel(profile=np.isin(key_psal, common_keys))

print(f"Common profiles: {ds_temp_common.profile.size}")

Common profiles: 340110


In [7]:
ds_temp_common["source"] = ds_temp_common["source"].astype(str)
ds_psal_common["source"] = ds_psal_common["source"].astype(str)

In [8]:
ds_temp_common.to_netcdf('/data0/user/aprigent/PROCESSED/level_3/temp_common_full.nc')
ds_psal_common.to_netcdf('/data0/user/aprigent/PROCESSED/level_3/psal_common_full.nc')